# Experiment: Oracle Beta Search

Objective:
- Use the same six application-style scenarios as the cross-method notebook.
- Initialize beta from a pilot-only rule, without the discrete scan.
- Run a non-exhaustive Optuna search over the q-target power `gamma` and swap size, deriving beta from the shared IID pilot for each trial.
- Evaluate each trial as multiple independent one-chain MCMC-IS runs at the HPO checkpoint.
- Minimize the per-chain RMSE relative to the exact permutation p-value.
- Compare the best oracle trial directly against the simple pilot initialization under that same per-chain objective.

In [1]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    _mcmc_eval_count,
    _steps_per_chain,
    build_beta_initialization,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_cross_method_saved_output,
    load_beta_sweep_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    plot_named_method_convergence,
    plot_named_method_max_budget,
    read_json,
    run_named_mcmc_checkpoint_study,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
    tune_samc_setup,
    write_json,
    write_jsonl,
    _effective_n_jobs,
    _iid_replicate_worker,
    _samc_replicate_worker,
    _try_make_process_pool,
)

pd.set_option("display.max_columns", 100)
project_root

PosixPath('/Users/noamchowers/Documents/University/Thesis/Code/MCMCIS')

In [2]:
import json
import numpy as np
import optuna
from optuna.samplers import TPESampler
from perm_pval.methods.beta_tuning import init_beta_from_iid_pilot

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

This notebook evaluates one final HPO budget per scenario, while saving intermediate checkpoints every `250k`.
Each Optuna trial consists of multiple independent one-chain MCMC-IS runs at that checkpoint.
The objective is the RMSE across those one-chain estimates.
`HPO_CHAIN_BUDGET` is the budget of each one-chain estimator.
There is no separate recheck stage and no separate final rerun of the winning configuration.

In [3]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "mcmcis_beta_notebook"

SCENARIO_GROUP = "threshold_suite"
SCENARIO_KEYS_OVERRIDE = [
    "gwas_additive_score_ultra_n100",
    "poisson_diffmeans_hep_ultra_n200",
    "gwas_additive_score_sig_n100",
    "poisson_diffmeans_hep_sig_n200",
    "gwas_additive_score_above_n100",
    "poisson_diffmeans_hep_above_n200",
]
HPO_CHAIN_COUNT = 40
HPO_N_JOBS = 6
HPO_CHAIN_BUDGET = 5_000_000 if not FAST_MODE else 200_000
HPO_CHECKPOINT_STEP = 250_000 if not FAST_MODE else 50_000
HPO_ESTIMATION_POINTS = tuple(range(HPO_CHECKPOINT_STEP, HPO_CHAIN_BUDGET + HPO_CHECKPOINT_STEP, HPO_CHECKPOINT_STEP))
HPO_OBJECTIVE_CHECKPOINT = HPO_CHAIN_BUDGET
HPO_OBJECTIVE_METRIC = "rmse"
HPO_GAMMA_LADDER = (
    0.10, 0.15, 0.20, 0.25, 0.3333,
    0.375, 0.40, 0.45, 0.50, 0.55,
)
HPO_SWAP_CHOICES_GRID = tuple(range(1, 11))
HPO_N_TRIALS = 15 if not FAST_MODE else 5
CANONICAL_GAMMA_SCAN = tuple(round(float(v), 4) for v in np.linspace(0.001, 1.0, 1000))
HPO_SWAP_CHOICES_DEFAULT = HPO_SWAP_CHOICES_GRID
HPO_SWAP_CHOICES_BY_SETTING = {
    "gwas_threshold_suite": HPO_SWAP_CHOICES_GRID,
    "hep_threshold_suite": HPO_SWAP_CHOICES_GRID,
}
DEFAULT_BASELINE_SWAP_BY_SETTING = {
    "gwas_threshold_suite": 2,
    "hep_threshold_suite": 5,
}
MIN_TAIL_STATES = 2
BASE_SEED = 54_321

init_cfg = MCMCWorkflowConfig(
    use_true_p0_for_q_target=False,
    p0_guess=1e-8,
    pilot_samples=200_000 if not FAST_MODE else 1_000,
    scale_method="sd",
    beta_max_init=1e6,
    chains=2,
    thin=1,
    estimate_variance=False,
    proposal_size=1,
)

hpo_eval_cfg = BetaSweepStudyConfig(
    estimation_points=HPO_ESTIMATION_POINTS,
    repeats=HPO_CHAIN_COUNT,
    beta_multipliers=(1.0,),
    chains=1,
    thin=1,
    estimate_variance=False,
    chain_n_jobs=1,
    proposal_size=1,
    base_seed=BASE_SEED,
    n_jobs=HPO_N_JOBS,
)

def search_reference_p0_for_scenario(scenario) -> float:
    return float(scenario.exact_p)


def canonical_threshold_for_scenario(scenario) -> float | None:
    threshold = scenario.extra.get("known_significance_threshold")
    if threshold is not None:
        threshold_f = float(threshold)
        if threshold_f == threshold_f and threshold_f > 0.0:
            return threshold_f
    return None


def beta_from_gamma_reference(
    *,
    gamma: float,
    p0_reference: float,
    pilot_t: np.ndarray,
    sigma_t: float,
    problem,
    beta_max: float,
) -> dict[str, float]:
    gamma_f = float(gamma)
    q_target = float(p0_reference ** gamma_f)
    beta = float(
        init_beta_from_iid_pilot(
            pilot_T=pilot_t,
            T_obs=problem.t_obs,
            sigma_T=sigma_t,
            p0=float(p0_reference),
            q_target=q_target,
            beta_max=float(beta_max),
        )
    )
    return {
        "gamma": gamma_f,
        "q_target": q_target,
        "beta": beta,
    }


def implied_gamma_for_beta_under_reference(
    *,
    target_beta: float,
    p0_reference: float | None,
    pilot_t: np.ndarray,
    sigma_t: float,
    problem,
    beta_max: float,
    gamma_grid: tuple[float, ...],
) -> dict[str, float | int | None]:
    if p0_reference is None or not np.isfinite(float(p0_reference)) or float(p0_reference) <= 0.0:
        return {
            "gamma": None,
            "q_target": None,
            "beta": None,
            "beta_abs_error": None,
            "beta_log_error": None,
            "grid_size": int(len(gamma_grid)),
        }
    rows = [
        beta_from_gamma_reference(
            gamma=float(gamma),
            p0_reference=float(p0_reference),
            pilot_t=pilot_t,
            sigma_t=float(sigma_t),
            problem=problem,
            beta_max=float(beta_max),
        )
        for gamma in gamma_grid
    ]
    target_beta_f = float(target_beta)
    eps = 1e-12
    best = min(
        rows,
        key=lambda row: abs(np.log(max(float(row["beta"]), eps)) - np.log(max(target_beta_f, eps))),
    )
    best_beta = float(best["beta"])
    return {
        "gamma": float(best["gamma"]),
        "q_target": float(best["q_target"]),
        "beta": best_beta,
        "beta_abs_error": float(abs(best_beta - target_beta_f)),
        "beta_log_error": float(abs(np.log(max(best_beta, eps)) - np.log(max(target_beta_f, eps)))),
        "grid_size": int(len(gamma_grid)),
    }


def swap_choices_for_scenario(scenario) -> tuple[int, ...]:
    setting_key = str(scenario.extra.get("application_setting_key", ""))
    if setting_key in HPO_SWAP_CHOICES_BY_SETTING:
        return tuple(int(v) for v in HPO_SWAP_CHOICES_BY_SETTING[setting_key])
    return tuple(int(v) for v in HPO_SWAP_CHOICES_DEFAULT)


def baseline_swap_for_scenario(scenario) -> int:
    setting_key = str(scenario.extra.get("application_setting_key", ""))
    if setting_key in DEFAULT_BASELINE_SWAP_BY_SETTING:
        return int(DEFAULT_BASELINE_SWAP_BY_SETTING[setting_key])
    return int(swap_choices_for_scenario(scenario)[0])


def trial_objective_value(summary_row: dict, metric: str) -> float:
    metric_val = float(summary_row[metric])
    if not np.isfinite(metric_val):
        return float("inf")
    return metric_val


print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_GROUP": SCENARIO_GROUP,
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "HPO_CHAIN_BUDGET": HPO_CHAIN_BUDGET,
    "HPO_CHECKPOINT_STEP": HPO_CHECKPOINT_STEP,
    "HPO_OBJECTIVE_CHECKPOINT": HPO_OBJECTIVE_CHECKPOINT,
    "HPO_N_CHECKPOINTS": len(HPO_ESTIMATION_POINTS),
    "HPO_N_TRIALS": HPO_N_TRIALS,
    "HPO_OBJECTIVE_METRIC": HPO_OBJECTIVE_METRIC,
    "HPO_GAMMA_LADDER": list(HPO_GAMMA_LADDER),
    "HPO_SWAP_CHOICES_GRID": list(HPO_SWAP_CHOICES_GRID),
    "HPO_CANDIDATE_GRID_SIZE": int(len(HPO_GAMMA_LADDER) * len(HPO_SWAP_CHOICES_GRID)),
    "HPO_EXHAUSTIVE_GRID": False,
    "HPO_SWAP_CHOICES_BY_SETTING": HPO_SWAP_CHOICES_BY_SETTING,
    "HPO_ONE_CHAIN_RUNS_PER_TRIAL": int(hpo_eval_cfg.repeats),
    "HPO_CHAIN_BUDGET_PER_RUN": int(HPO_CHAIN_BUDGET),
    "HPO_TOP_LEVEL_N_JOBS": int(hpo_eval_cfg.n_jobs),
    "HPO_CHAIN_N_JOBS": int(hpo_eval_cfg.chain_n_jobs),
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
    "REFERENCE_P0_MODE": "oracle_exact_p__simple_known_threshold_else_exact_p",
    "CANONICAL_GAMMA_SCAN_POINTS": int(len(CANONICAL_GAMMA_SCAN)),
}, indent=2))

{
  "FAST_MODE": false,
  "SCENARIO_GROUP": "threshold_suite",
  "SCENARIO_KEYS_OVERRIDE": [
    "gwas_additive_score_ultra_n100",
    "poisson_diffmeans_hep_ultra_n200",
    "gwas_additive_score_sig_n100",
    "poisson_diffmeans_hep_sig_n200",
    "gwas_additive_score_above_n100",
    "poisson_diffmeans_hep_above_n200"
  ],
  "HPO_CHAIN_BUDGET": 5000000,
  "HPO_CHECKPOINT_STEP": 250000,
  "HPO_OBJECTIVE_CHECKPOINT": 5000000,
  "HPO_N_CHECKPOINTS": 20,
  "HPO_N_TRIALS": 15,
  "HPO_OBJECTIVE_METRIC": "rmse",
  "HPO_GAMMA_LADDER": [
    0.1,
    0.15,
    0.2,
    0.25,
    0.3333,
    0.375,
    0.4,
    0.45,
    0.5,
    0.55
  ],
  "HPO_SWAP_CHOICES_GRID": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10
  ],
  "HPO_CANDIDATE_GRID_SIZE": 100,
  "HPO_EXHAUSTIVE_GRID": false,
  "HPO_SWAP_CHOICES_BY_SETTING": {
    "gwas_threshold_suite": [
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10
    ],
    "hep_threshold_suite": [

## Load Scenarios

In [4]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None if SCENARIO_KEYS_OVERRIDE is not None else SCENARIO_GROUP,
    min_tail_states=MIN_TAIL_STATES,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "oracle_beta_search") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "tail_hits": s.exact_tail_hits,
        "n_perm": s.exact_n_perm,
        "exact_method": s.exact_method,
        "family": s.portfolio.get("family"),
        "rarity_band": s.portfolio.get("rarity_band"),
        "difficulty": s.portfolio.get("expected_difficulty"),
    }
    for s in scenarios
])

,scenario,exact_p,tail_hits,n_perm,exact_method,family,rarity_band,difficulty
0,gwas_additive_score_ultra_n100,1.144092e-10,11542898647056384036,100891344545564193334812497256,LinearStatisticDPSolver,gwas_additive_score,ultra_rare,moderate
1,poisson_diffmeans_hep_ultra_n200,3.759363e-09,3404047068642153314749889029354901204235267136...,9054851465610328116540417707748416387450458967...,LinearStatisticDPSolver,poisson_count,ultra_rare,moderate
2,gwas_additive_score_sig_n100,4.370469e-08,4409425215945901258536,100891344545564193334812497256,LinearStatisticDPSolver,gwas_additive_score,extreme,moderate
3,poisson_diffmeans_hep_sig_n200,2.533610e-07,2294146183289570997753507502932653227347376762...,9054851465610328116540417707748416387450458967...,LinearStatisticDPSolver,poisson_count,extreme,moderate
4,gwas_additive_score_above_n100,5.120947e-07,51665924064804456121416,100891344545564193334812497256,LinearStatisticDPSolver,gwas_additive_score,extreme,moderate
5,poisson_diffmeans_hep_above_n200,7.668784e-07,6943969698094447475781867186094253216914531992...,9054851465610328116540417707748416387450458967...,LinearStatisticDPSolver,poisson_count,extreme,moderate


## Run Oracle Beta HPO

In [ ]:
oracle_results = {}

for scenario_idx, scenario in enumerate(scenarios):
    scenario_seed = BASE_SEED + 50_000 * scenario_idx
    search_reference_p0 = search_reference_p0_for_scenario(scenario)
    canonical_threshold_p0 = canonical_threshold_for_scenario(scenario)
    simple_reference_p0 = (
        float(canonical_threshold_p0)
        if canonical_threshold_p0 is not None
        else float(search_reference_p0)
    )
    init_payload = build_beta_initialization(
        scenario.problem,
        scenario.exact_p,
        init_cfg,
        seed=scenario_seed,
        p0_reference=simple_reference_p0,
    )
    beta_init = float(init_payload["beta0_laplace"])
    sigma_t = float(init_payload["sigma_t"])
    pilot_t_shared = np.asarray(init_payload["pilot_t"], dtype=float)
    simple_gamma = float(init_cfg.d_alpha)
    swap_choices = swap_choices_for_scenario(scenario)
    baseline_swap = baseline_swap_for_scenario(scenario)
    trial_rows = []
    trial_checkpoint_rows = []
    trial_record_rows = []
    scenario_dir = (run_dir / scenario.key) if (SAVE_OUTPUTS and run_dir is not None) else None
    if scenario_dir is not None:
        scenario_dir.mkdir(parents=True, exist_ok=True)

    print(json.dumps({
        "scenario": scenario.key,
        "exact_p": scenario.exact_p,
        "application_setting_key": scenario.extra.get("application_setting_key"),
        "known_significance_threshold": scenario.extra.get("known_significance_threshold"),
        "search_reference_p0": search_reference_p0,
        "simple_reference_p0": simple_reference_p0,
        "canonical_threshold_p0": canonical_threshold_p0,
        "beta0_formula": init_payload["beta0_formula"],
        "beta0_laplace": beta_init,
        "simple_gamma": simple_gamma,
        "q_target": init_payload["q_target"],
        "sigma_t": sigma_t,
        "pilot_samples": int(init_cfg.pilot_samples),
        "swap_choices": swap_choices,
        "baseline_swap": baseline_swap,
    }, indent=2))

    init_eval = run_named_mcmc_checkpoint_study(
        scenario.problem,
        scenario.exact_p,
        config_specs=[
            {
                "label": "simple_init",
                "config_id": "simple_init",
                "beta": float(beta_init),
                "proposal_size": int(baseline_swap),
                "source": "pilot_init",
            }
        ],
        sigma_t=sigma_t,
        estimation_points=HPO_ESTIMATION_POINTS,
        repeats=int(hpo_eval_cfg.repeats),
        base_seed=BASE_SEED + 250_000 + 25_000 * scenario_idx,
        template_cfg=hpo_eval_cfg,
        n_jobs=int(hpo_eval_cfg.n_jobs),
    )
    init_summary_rows = [dict(row) for row in init_eval["summary"]]
    init_summary = next(row for row in init_summary_rows if int(row["checkpoint"]) == int(HPO_OBJECTIVE_CHECKPOINT))
    init_summary["label"] = "simple_init"
    init_summary["beta"] = float(beta_init)
    init_summary["gamma"] = float(simple_gamma)
    init_summary["q_target"] = float(simple_reference_p0 ** simple_gamma)
    init_summary["reference_p0"] = float(simple_reference_p0)
    init_summary["proposal_size"] = int(baseline_swap)
    init_summary["source"] = "pilot_init"
    init_summary_json = json.loads(pd.DataFrame([init_summary]).to_json(orient="records"))[0]

    def objective(trial):
        gamma = float(trial.suggest_categorical("gamma", list(HPO_GAMMA_LADDER)))
        proposal_size = int(trial.suggest_categorical("proposal_size", list(swap_choices)))
        beta_payload = beta_from_gamma_reference(
            gamma=gamma,
            p0_reference=float(search_reference_p0),
            pilot_t=pilot_t_shared,
            sigma_t=float(sigma_t),
            problem=scenario.problem,
            beta_max=float(init_cfg.beta_max_init),
        )
        q_target = float(beta_payload["q_target"])
        beta = float(beta_payload["beta"])
        trial_eval = run_named_mcmc_checkpoint_study(
            scenario.problem,
            scenario.exact_p,
            config_specs=[
                {
                    "label": "oracle_trial",
                    "config_id": f"trial_{trial.number}",
                    "beta": beta,
                    "proposal_size": proposal_size,
                    "source": "oracle_hpo",
                }
            ],
            sigma_t=sigma_t,
            estimation_points=HPO_ESTIMATION_POINTS,
            repeats=int(hpo_eval_cfg.repeats),
            base_seed=BASE_SEED + 1_000_000 * scenario_idx + 10_000 * trial.number,
            template_cfg=hpo_eval_cfg,
            n_jobs=int(hpo_eval_cfg.n_jobs),
        )
        summary_rows = [dict(row) for row in trial_eval["summary"]]
        summary_row = next(row for row in summary_rows if int(row["checkpoint"]) == int(HPO_OBJECTIVE_CHECKPOINT))
        objective_value = trial_objective_value(summary_row, HPO_OBJECTIVE_METRIC)
        for checkpoint_row in summary_rows:
            trial_checkpoint_rows.append(
                {
                    "trial_number": int(trial.number),
                    "scenario": scenario.key,
                    "gamma": gamma,
                    "q_target": q_target,
                    "beta": beta,
                    "proposal_size": proposal_size,
                    **checkpoint_row,
                }
            )
        for record_row in trial_eval["records"]:
            trial_record_rows.append(
                {
                    "trial_number": int(trial.number),
                    "scenario": scenario.key,
                    "gamma": gamma,
                    "q_target": q_target,
                    "beta": beta,
                    "proposal_size": proposal_size,
                    **dict(record_row),
                }
            )
        trial_payload = {
            "trial_number": int(trial.number),
            "scenario": scenario.key,
            "gamma": gamma,
            "q_target": q_target,
            "beta": beta,
            "proposal_size": proposal_size,
            "objective_metric": HPO_OBJECTIVE_METRIC,
            "objective_checkpoint": int(HPO_OBJECTIVE_CHECKPOINT),
            "objective_value": float(objective_value),
            "reference_p0": float(search_reference_p0),
            **summary_row,
        }
        trial_rows.append(trial_payload)
        if scenario_dir is not None:
            pd.DataFrame(trial_rows).to_json(
                scenario_dir / "hpo_trials.jsonl",
                orient="records",
                lines=True,
            )
            pd.DataFrame(trial_checkpoint_rows).to_json(
                scenario_dir / "hpo_trial_checkpoint_summaries.jsonl",
                orient="records",
                lines=True,
            )
            pd.DataFrame(trial_record_rows).to_json(
                scenario_dir / "hpo_trial_records.jsonl",
                orient="records",
                lines=True,
            )
            (scenario_dir / "hpo_status.json").write_text(
                json.dumps(
                    {
                        "scenario": scenario.key,
                        "completed_trials": len(trial_rows),
                        "target_trials": int(HPO_N_TRIALS),
                        "latest_trial_number": int(trial.number),
                        "objective": "per_chain_rmse",
                        "objective_checkpoint": int(HPO_OBJECTIVE_CHECKPOINT),
                    },
                    indent=2,
                ),
                encoding="utf-8",
            )
        for key, value in trial_payload.items():
            if isinstance(value, (str, int, float)) and not isinstance(value, bool):
                trial.set_user_attr(str(key), value)
        return float(objective_value)

    def persist_hpo_state(study, trial):
        if scenario_dir is None or len(study.trials) == 0:
            return
        best_trial = study.best_trial
        best_gamma = float(best_trial.params["gamma"])
        best_q_target = float(search_reference_p0 ** best_gamma)
        best_beta = float(best_trial.user_attrs["beta"])
        implied_canonical = implied_gamma_for_beta_under_reference(
            target_beta=best_beta,
            p0_reference=canonical_threshold_p0,
            pilot_t=pilot_t_shared,
            sigma_t=float(sigma_t),
            problem=scenario.problem,
            beta_max=float(init_cfg.beta_max_init),
            gamma_grid=CANONICAL_GAMMA_SCAN,
        )
        best_payload = {
            "scenario": scenario.key,
            "reference_p0": float(search_reference_p0),
            "simple_reference_p0": float(simple_reference_p0),
            "canonical_threshold_p0": (
                float(canonical_threshold_p0)
                if canonical_threshold_p0 is not None
                else None
            ),
            "beta_init": float(beta_init),
            "beta0_formula": float(init_payload["beta0_formula"]),
            "beta0_laplace": float(init_payload["beta0_laplace"]),
            "sigma_t": float(sigma_t),
            "best_gamma": best_gamma,
            "best_q_target": best_q_target,
            "best_beta": best_beta,
            "implied_gamma_canonical": implied_canonical["gamma"],
            "implied_q_target_canonical": implied_canonical["q_target"],
            "implied_beta_canonical": implied_canonical["beta"],
            "implied_beta_abs_error_canonical": implied_canonical["beta_abs_error"],
            "implied_beta_log_error_canonical": implied_canonical["beta_log_error"],
            "best_proposal_size": int(best_trial.params["proposal_size"]),
            "best_objective_value": float(study.best_value),
            "best_trial_number": int(best_trial.number),
            "completed_trials": int(len(study.trials)),
        }
        (scenario_dir / "best_config.partial.json").write_text(
            json.dumps(best_payload, indent=2),
            encoding="utf-8",
        )
        pd.DataFrame(
            [
                {
                    "trial_number": int(t.number),
                    "state": str(t.state),
                    "value": float(t.value) if t.value is not None else None,
                    **{str(k): v for k, v in t.params.items()},
                }
                for t in study.trials
            ]
        ).to_json(
            scenario_dir / "optuna_trials.json",
            orient="records",
            indent=2,
        )

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=BASE_SEED + 10_000 * scenario_idx + 7),
        study_name=f"{scenario.key}_oracle_beta",
    )
    study.optimize(
        objective,
        n_trials=HPO_N_TRIALS,
        show_progress_bar=False,
            callbacks=[persist_hpo_state],
        )

    best_trial = study.best_trial
    best_gamma = float(best_trial.params["gamma"])
    best_q_target = float(search_reference_p0 ** best_gamma)
    best_beta = float(best_trial.user_attrs["beta"])
    best_proposal_size = int(best_trial.params["proposal_size"])
    best_source = "oracle_hpo"
    best_trial_number = int(best_trial.number)
    implied_canonical = implied_gamma_for_beta_under_reference(
        target_beta=best_beta,
        p0_reference=canonical_threshold_p0,
        pilot_t=pilot_t_shared,
        sigma_t=float(sigma_t),
        problem=scenario.problem,
        beta_max=float(init_cfg.beta_max_init),
        gamma_grid=CANONICAL_GAMMA_SCAN,
    )
    best_trial_payload = dict(best_trial.user_attrs)
    best_trial_payload.update(
        {
            "trial_number": int(best_trial.number),
            "gamma": best_gamma,
            "q_target": best_q_target,
            "beta": float(best_beta),
            "proposal_size": int(best_proposal_size),
            "objective_metric": HPO_OBJECTIVE_METRIC,
            "objective_checkpoint": int(HPO_OBJECTIVE_CHECKPOINT),
            "objective_value": float(study.best_value),
            "reference_p0": float(search_reference_p0),
            "canonical_threshold_p0": (
                float(canonical_threshold_p0)
                if canonical_threshold_p0 is not None
                else None
            ),
            "implied_gamma_canonical": implied_canonical["gamma"],
            "implied_q_target_canonical": implied_canonical["q_target"],
            "implied_beta_canonical": implied_canonical["beta"],
            "label": "oracle",
        }
    )
    best_trial_payload_json = json.loads(pd.DataFrame([best_trial_payload]).to_json(orient="records"))[0]

    best_config = {
        "scenario": scenario.key,
        "reference_p0": float(search_reference_p0),
        "simple_reference_p0": float(simple_reference_p0),
        "canonical_threshold_p0": (
            float(canonical_threshold_p0)
            if canonical_threshold_p0 is not None
            else None
        ),
        "beta_init": float(beta_init),
        "simple_gamma": float(simple_gamma),
        "beta0_formula": float(init_payload["beta0_formula"]),
        "beta0_laplace": float(init_payload["beta0_laplace"]),
        "sigma_t": float(sigma_t),
        "best_gamma": best_gamma,
        "best_q_target": best_q_target,
        "best_beta": best_beta,
        "implied_gamma_canonical": implied_canonical["gamma"],
        "implied_q_target_canonical": implied_canonical["q_target"],
        "implied_beta_canonical": implied_canonical["beta"],
        "implied_beta_abs_error_canonical": implied_canonical["beta_abs_error"],
        "implied_beta_log_error_canonical": implied_canonical["beta_log_error"],
        "best_proposal_size": int(best_proposal_size),
        "best_objective_value": float(study.best_value),
        "best_trial_number": int(best_trial_number),
        "n_trials": int(len(study.trials)),
        "selection_source": best_source,
    }

    oracle_results[scenario.key] = {
        "best_config": best_config,
        "trial_rows": list(trial_rows),
        "trial_checkpoint_rows": list(trial_checkpoint_rows),
        "trial_record_rows": list(trial_record_rows),
        "init_summary": dict(init_summary_json),
        "best_trial_summary": dict(best_trial_payload_json),
        "init_payload": {
            key: value
            for key, value in init_payload.items()
            if key != "pilot_t"
        },
    }

    if SAVE_OUTPUTS and run_dir is not None:
        pd.DataFrame(trial_rows).to_json(
            scenario_dir / "hpo_trials.jsonl",
            orient="records",
            lines=True,
        )
        pd.DataFrame(trial_checkpoint_rows).to_json(
            scenario_dir / "hpo_trial_checkpoint_summaries.jsonl",
            orient="records",
            lines=True,
        )
        pd.DataFrame(trial_record_rows).to_json(
            scenario_dir / "hpo_trial_records.jsonl",
            orient="records",
            lines=True,
        )
        pd.DataFrame(init_eval["records"]).to_json(
            scenario_dir / "simple_init_records.jsonl",
            orient="records",
            lines=True,
        )
        pd.DataFrame(init_summary_rows).to_json(
            scenario_dir / "simple_init_checkpoint_summaries.json",
            orient="records",
            indent=2,
        )
        (scenario_dir / "simple_init_summary.json").write_text(
            json.dumps(init_summary_json, indent=2),
            encoding="utf-8",
        )
        (scenario_dir / "oracle_best_trial.json").write_text(
            json.dumps(best_trial_payload_json, indent=2),
            encoding="utf-8",
        )
        (scenario_dir / "best_config.json").write_text(
            json.dumps(best_config, indent=2),
            encoding="utf-8",
        )
        (scenario_dir / "metadata.json").write_text(
            json.dumps(
                {
                    "scenario": scenario.key,
                    "scenario_display": scenario.description,
                    "exact_p": float(scenario.exact_p),
                    "known_significance_threshold": scenario.extra.get("known_significance_threshold"),
                    "application_setting_key": scenario.extra.get("application_setting_key"),
                    "reference_p0": float(search_reference_p0),
                    "reference_p0_mode": "exact_p",
                    "simple_reference_p0": float(simple_reference_p0),
                    "simple_reference_p0_mode": (
                        "known_significance_threshold"
                        if canonical_threshold_p0 is not None
                        else "exact_p"
                    ),
                    "canonical_threshold_p0": (
                        float(canonical_threshold_p0)
                        if canonical_threshold_p0 is not None
                        else None
                    ),
                    "init_payload": oracle_results[scenario.key]["init_payload"],
                    "hpo_settings": {
                        "n_trials": int(HPO_N_TRIALS),
                        "objective_definition": "rmse_across_one_chain_estimates",
                        "objective_metric": str(HPO_OBJECTIVE_METRIC),
                        "gamma_ladder": [float(v) for v in HPO_GAMMA_LADDER],
                        "canonical_gamma_scan": {
                            "min": float(CANONICAL_GAMMA_SCAN[0]),
                            "max": float(CANONICAL_GAMMA_SCAN[-1]),
                            "n_points": int(len(CANONICAL_GAMMA_SCAN)),
                        },
                        "swap_choices": [int(v) for v in swap_choices],
                        "one_chain_runs_per_trial": int(hpo_eval_cfg.repeats),
                        "chain_budget": int(HPO_CHAIN_BUDGET),
                        "checkpoint_step": int(HPO_CHECKPOINT_STEP),
                        "estimation_points": [int(v) for v in HPO_ESTIMATION_POINTS],
                        "objective_checkpoint": int(HPO_OBJECTIVE_CHECKPOINT),
                        "chain_n_jobs": int(hpo_eval_cfg.chain_n_jobs),
                        "top_level_n_jobs": int(hpo_eval_cfg.n_jobs),
                        "default_gamma": float(simple_gamma),
                    },
                },
                indent=2,
            ),
            encoding="utf-8",
        )

    best_row = pd.DataFrame([best_config])
    comparison_df = pd.DataFrame([
        dict(init_summary_json),
        dict(best_trial_payload_json),
    ]).sort_values("label")
    display(best_row)
    display(comparison_df[[
        "checkpoint",
        "label",
        "mean_estimate",
        "rmse",
        "mean_abs_log10_error",
        "mean_variance_estimate",
        "mean_q_tilt_tail_share",
        "mean_ess",
        "mean_acceptance_rate",
        "mean_weight_cv",
    ]])

{
  "scenario": "gwas_additive_score_ultra_n100",
  "exact_p": 1.1440920625102207e-10,
  "application_setting_key": "gwas_threshold_suite",
  "known_significance_threshold": 5e-08,
  "search_reference_p0": 1.1440920625102207e-10,
  "simple_reference_p0": 5e-08,
  "canonical_threshold_p0": 5e-08,
  "beta0_formula": 4.784479027647473,
  "beta0_laplace": 2.3076171875,
  "simple_gamma": 0.3333333333333333,
  "q_target": 0.0036840314986403876,
  "sigma_t": 2.5946948819866384,
  "pilot_samples": 200000,
  "swap_choices": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10
  ],
  "baseline_swap": 2
}


[I 2026-05-28 09:40:40,054] A new study created in memory with name: gwas_additive_score_ultra_n100_oracle_beta
[I 2026-05-28 09:58:42,707] Trial 0 finished with value: 1.425020825928435e-10 and parameters: {'gamma': 0.15, 'proposal_size': 9}. Best is trial 0 with value: 1.425020825928435e-10.
[I 2026-05-28 10:17:03,117] Trial 1 finished with value: 3.852261935380904e-11 and parameters: {'gamma': 0.25, 'proposal_size': 3}. Best is trial 1 with value: 3.852261935380904e-11.
[I 2026-05-28 10:35:17,583] Trial 2 finished with value: 7.40339392525949e-11 and parameters: {'gamma': 0.375, 'proposal_size': 9}. Best is trial 1 with value: 3.852261935380904e-11.
[I 2026-05-28 10:53:31,711] Trial 3 finished with value: 5.714351253244395e-11 and parameters: {'gamma': 0.5, 'proposal_size': 3}. Best is trial 1 with value: 3.852261935380904e-11.
[I 2026-05-28 11:11:50,369] Trial 4 finished with value: 3.903891488783823e-10 and parameters: {'gamma': 0.1, 'proposal_size': 6}. Best is trial 1 with value

,scenario,reference_p0,simple_reference_p0,canonical_threshold_p0,beta_init,simple_gamma,beta0_formula,beta0_laplace,sigma_t,best_gamma,best_q_target,best_beta,implied_gamma_canonical,implied_q_target_canonical,implied_beta_canonical,implied_beta_abs_error_canonical,implied_beta_log_error_canonical,best_proposal_size,best_objective_value,best_trial_number,n_trials,selection_source
0,gwas_additive_score_ultra_n100,1.144092e-10,5.000000e-08,5.000000e-08,2.307617,0.333333,4.784479,2.307617,2.594695,0.4,0.000106,3.025879,0.183,0.046122,3.025879,0.0,0.0,2,2.080174e-11,14,15,oracle_hpo


,checkpoint,label,mean_estimate,rmse,mean_abs_log10_error,mean_variance_estimate,mean_q_tilt_tail_share,mean_ess,mean_acceptance_rate,mean_weight_cv
1,5000000,oracle,1.000000e-10,0.000000e+00,0.060707,None,0.000103,3225.920793,0.593052,37.662762
0,5000000,simple_init,1.000000e-10,1.000000e-10,0.187230,None,0.000008,27959.213170,0.678733,12.855918


{
  "scenario": "poisson_diffmeans_hep_ultra_n200",
  "exact_p": 3.759362681508888e-09,
  "application_setting_key": "hep_threshold_suite",
  "known_significance_threshold": 3e-07,
  "search_reference_p0": 3.759362681508888e-09,
  "simple_reference_p0": 3e-07,
  "canonical_threshold_p0": 3e-07,
  "beta0_formula": 4.404431449518767,
  "beta0_laplace": 2.22216796875,
  "simple_gamma": 0.3333333333333333,
  "q_target": 0.006694329500821697,
  "sigma_t": 0.2393254338461051,
  "pilot_samples": 200000,
  "swap_choices": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10
  ],
  "baseline_swap": 5
}


...

[I 2026-05-29 08:20:37,959] A new study created in memory with name: gwas_additive_score_above_n100_oracle_beta
[I 2026-05-29 08:40:16,904] Trial 0 finished with value: 5.11749108178377e-08 and parameters: {'gamma': 0.55, 'proposal_size': 8}. Best is trial 0 with value: 5.11749108178377e-08.
[I 2026-05-29 09:00:13,604] Trial 1 finished with value: 2.834896302447458e-08 and parameters: {'gamma': 0.3333, 'proposal_size': 9}. Best is trial 1 with value: 2.834896302447458e-08.
[I 2026-05-29 09:20:22,582] Trial 2 finished with value: 2.9210972613092314e-08 and parameters: {'gamma': 0.3333, 'proposal_size': 10}. Best is trial 1 with value: 2.834896302447458e-08.
[I 2026-05-29 09:40:00,373] Trial 3 finished with value: 6.211900264935604e-07 and parameters: {'gamma': 0.1, 'proposal_size': 1}. Best is trial 1 with value: 2.834896302447458e-08.
[I 2026-05-29 10:00:02,060] Trial 4 finished with value: 2.0563896882099457e-07 and parameters: {'gamma': 0.15, 'proposal_size': 2}. Best is trial 1 with

,scenario,reference_p0,simple_reference_p0,canonical_threshold_p0,beta_init,simple_gamma,beta0_formula,beta0_laplace,sigma_t,best_gamma,best_q_target,best_beta,implied_gamma_canonical,implied_q_target_canonical,implied_beta_canonical,implied_beta_abs_error_canonical,implied_beta_log_error_canonical,best_proposal_size,best_objective_value,best_trial_number,n_trials,selection_source
0,gwas_additive_score_above_n100,5.120947e-07,5.000000e-08,5.000000e-08,3.605469,0.333333,3.805884,3.605469,2.605118,0.375,0.004375,2.535156,0.462,0.000424,2.532227,0.00293,0.001156,6,2.137570e-08,11,15,oracle_hpo


,checkpoint,label,mean_estimate,rmse,mean_abs_log10_error,mean_variance_estimate,mean_q_tilt_tail_share,mean_ess,mean_acceptance_rate,mean_weight_cv
1,5000000,oracle,5.168000e-07,2.140000e-08,0.014787,None,0.004444,11350.926931,0.405809,20.925786
0,5000000,simple_init,4.943000e-07,9.350000e-08,0.063766,None,0.037017,523.587934,0.534972,128.832235


{
  "scenario": "poisson_diffmeans_hep_above_n200",
  "exact_p": 7.668783661959717e-07,
  "application_setting_key": "hep_threshold_suite",
  "known_significance_threshold": 3e-07,
  "search_reference_p0": 7.668783661959717e-07,
  "simple_reference_p0": 3e-07,
  "canonical_threshold_p0": 3e-07,
  "beta0_formula": 3.7524575456616787,
  "beta0_laplace": 3.13671875,
  "simple_gamma": 0.3333333333333333,
  "q_target": 0.006694329500821697,
  "sigma_t": 0.24515828115679492,
  "pilot_samples": 200000,
  "swap_choices": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10
  ],
  "baseline_swap": 5
}


[I 2026-05-29 13:42:39,803] A new study created in memory with name: poisson_diffmeans_hep_above_n200_oracle_beta
[I 2026-05-29 14:08:11,922] Trial 0 finished with value: 1.0152194317345653e-06 and parameters: {'gamma': 0.1, 'proposal_size': 6}. Best is trial 0 with value: 1.0152194317345653e-06.
[I 2026-05-29 14:33:29,379] Trial 1 finished with value: 7.262622283488554e-08 and parameters: {'gamma': 0.55, 'proposal_size': 1}. Best is trial 1 with value: 7.262622283488554e-08.
[I 2026-05-29 14:58:34,575] Trial 2 finished with value: 3.175895903112369e-08 and parameters: {'gamma': 0.45, 'proposal_size': 7}. Best is trial 2 with value: 3.175895903112369e-08.
[I 2026-05-29 15:22:02,206] Trial 3 finished with value: 3.829880576637821e-08 and parameters: {'gamma': 0.3333, 'proposal_size': 6}. Best is trial 2 with value: 3.175895903112369e-08.
[I 2026-05-29 15:45:14,456] Trial 4 finished with value: 1.0895621913216947e-07 and parameters: {'gamma': 0.25, 'proposal_size': 9}. Best is trial 2 wi

## Review Saved Outputs

In [1]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    display(pd.DataFrame([
        {
            "scenario": key,
            "simple_gamma": payload["best_config"]["simple_gamma"],
            "simple_beta": payload["best_config"]["beta_init"],
            "best_gamma": payload["best_config"]["best_gamma"],
            "best_beta": payload["best_config"]["best_beta"],
            "best_proposal_size": payload["best_config"]["best_proposal_size"],
            "simple_rmse": payload["init_summary"]["rmse"],
            "oracle_rmse": payload["best_trial_summary"]["rmse"],
            "best_objective_value": payload["best_config"]["best_objective_value"],
        }
        for key, payload in oracle_results.items()
    ]))
else:
    print("SAVE_OUTPUTS=False, so no saved outputs to display.")

NameError: name 'SAVE_OUTPUTS' is not defined

## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_BETA_DIR = None
# # Example:
# # RELOAD_BETA_DIR = project_root / "results" / "mcmcis_beta_notebook" / "20260411_120000_oracle_beta_search" / "gwas_additive_score_sig_n100"

# if RELOAD_BETA_DIR is not None:
#     print(json.loads((RELOAD_BETA_DIR / "best_config.json").read_text()))
#     display(pd.DataFrame([
#         json.loads((RELOAD_BETA_DIR / "simple_init_summary.json").read_text()),
#         json.loads((RELOAD_BETA_DIR / "oracle_best_trial.json").read_text()),
#     ]))
#     display(pd.read_json(RELOAD_BETA_DIR / "hpo_trials.jsonl", lines=True).sort_values("objective_value").head())
# else:
#     print("Set RELOAD_BETA_DIR to a saved oracle-beta directory to inspect saved results from disk only.")